# Step 6: Random Forest Baseline

## 1. Objective

Train and evaluate Random Forest baselines to see how a nonlinear, interaction-capable tree ensemble behaves on the same causal feature set used for Logistic Regression (Step 5). Still baseline modeling: no exhaustive hyperparameter search, and no final model selection. The Logistic Regression artifacts from Step 5 are not modified anywhere in this notebook.

## 2. Load chronological datasets

Same `train.parquet`/`validation.parquet`/`test.parquet` from Step 4 -- no resplitting.

In [ ]:
import sys
sys.path.append("..")

import json
import joblib
import numpy as np
import pandas as pd

from src.model_utils import (
    load_split_data,
    validate_model_features,
    build_random_forest,
    train_model,
    tree_depth_summary,
    evaluate_classifier,
    evaluate_at_k,
    extract_tree_feature_importance,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

data = load_split_data()
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_validation"], data["y_validation"]
X_test, y_test = data["X_test"], data["y_test"]
feature_columns = data["feature_columns"]

print("X_train:", X_train.shape, " X_validation:", X_val.shape, " X_test:", X_test.shape)
for name, y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: n={len(y):,}, fraud={int(y.sum()):,}, rate={y.mean()*100:.4f}%")

## 3. Validate model features

In [ ]:
print("Feature count:", len(feature_columns), "(expected 42)")
print("Identical columns across all 3 splits:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))
for split_name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    checks = validate_model_features(X, feature_columns)
    assert all(checks.values()), f"{split_name} failed: {checks}"
    print(f"{split_name}: {checks}")

## 4. Why Random Forest / why no scaling

Logistic Regression assumes a linear relationship in log-odds between (scaled) features and the outcome. Real fraud patterns can involve interactions Logistic Regression cannot represent directly -- e.g. amount **combined with** transaction type, or sender balance **combined with** amount, rather than either alone. A tree ensemble splits on raw feature values and can capture such interactions and nonlinear thresholds without any manual feature-crossing.

**No `StandardScaler` is used here.** Tree splits only compare a feature's values to a threshold ("is `amount_to_sender_balance` > 0.98?"), which is invariant to monotonic rescaling -- scaling would change nothing about the splits found and just adds unnecessary preprocessing. This is a deliberate, documented difference from Step 5's `Pipeline(StandardScaler -> LogisticRegression)`.

## 5. Random Forest configuration

In [ ]:
config = dict(n_estimators=200, max_depth=20, min_samples_split=2, min_samples_leaf=1,
              max_features="sqrt", n_jobs=-1, random_state=42)
print("Configuration:", config)
print("\nRationale: this is a commonly-used, computationally reasonable STARTING configuration, not a")
print("tuned one. max_depth=20 bounds memory/training time on 4.46M rows; max_features='sqrt' (~6 of 42")
print("features per split) decorrelates trees; n_jobs=-1 uses all available CPU cores. No GridSearchCV or")
print("RandomizedSearchCV is used anywhere in this step -- these settings are fixed a priori.")

## 6. Model C -- unweighted Random Forest

In [ ]:
rf_c = build_random_forest(class_weight=None, **config)
info_c = train_model(rf_c, X_train, y_train)
depth_c = tree_depth_summary(info_c["pipeline"])
print(f"Model C: training_time_sec={info_c['training_time_sec']:.1f}")
print(f"Tree depth summary: {depth_c}")
print("\nFit ONLY on (X_train, y_train).")

## 7. Model D -- class-weighted Random Forest (`balanced_subsample`)

In [ ]:
rf_d = build_random_forest(class_weight="balanced_subsample", **config)
info_d = train_model(rf_d, X_train, y_train)
depth_d = tree_depth_summary(info_d["pipeline"])
print(f"Model D: training_time_sec={info_d['training_time_sec']:.1f}")
print(f"Tree depth summary: {depth_d}")
print("\nFit ONLY on (X_train, y_train).")

## 8. Validation evaluation (threshold = 0.5)

In [ ]:
fitted = {"model_c_unweighted": info_c, "model_d_balanced_subsample": info_d}
depths = {"model_c_unweighted": depth_c, "model_d_balanced_subsample": depth_d}
results = {}

for name, info in fitted.items():
    proba_val = info["pipeline"].predict_proba(X_val)[:, 1]
    info["proba_val"] = proba_val
    val_metrics = evaluate_classifier(y_val, proba_val, threshold=0.5)
    results.setdefault(name, {})["validation_metrics"] = val_metrics
    print(f"{name} -- VALIDATION @ threshold=0.5: {val_metrics}")
    print()

## 9. Test evaluation -- FINAL TEST RESULTS (NOT used for model selection)

In [ ]:
for name, info in fitted.items():
    proba_test = info["pipeline"].predict_proba(X_test)[:, 1]
    info["proba_test"] = proba_test
    test_metrics = evaluate_classifier(y_test, proba_test, threshold=0.5)
    results[name]["test_metrics"] = test_metrics
    print(f"{name} -- FINAL TEST RESULTS @ threshold=0.5: {test_metrics}")
    print()

## 10. Precision@K / Recall@K

In [ ]:
k_values = [100, 500, 1000, 5000, 10000]
for name, info in fitted.items():
    at_k_val = evaluate_at_k(y_val, info["proba_val"], k_values)
    at_k_test = evaluate_at_k(y_test, info["proba_test"], k_values)
    results[name]["at_k_validation"] = at_k_val.to_dict(orient="records")
    results[name]["at_k_test"] = at_k_test.to_dict(orient="records")
    print(f"{name} -- VALIDATION:\n{at_k_val.to_string(index=False)}")
    print(f"\n{name} -- TEST:\n{at_k_test.to_string(index=False)}\n")

## 11. Confusion matrices (threshold = 0.5)

In [ ]:
for name in fitted:
    for split in ["validation", "test"]:
        cm = results[name][f"{split}_metrics"]["confusion_matrix"]
        print(f"{name} -- {split}: TN={cm['tn']:,} FP={cm['fp']:,} FN={cm['fn']:,} TP={cm['tp']:,}")

## 12. Feature importance

Impurity-based `feature_importances_`: how much each feature contributed to reducing impurity across this fitted forest's splits. This is **not** a causal measure, can be **biased toward continuous/high-cardinality features** over binary ones, and gets **split across correlated/redundant features** -- exactly what we see below with `type_TRANSFER`/`is_transfer` and `type_CASH_OUT`/`is_cash_out`, which carry the same signal but each show only part of the combined importance. No SHAP is used here (a separate future step); we do not call any of this a universal fraud driver.

In [ ]:
importances = {}
for name, info in fitted.items():
    imp = extract_tree_feature_importance(info["pipeline"], feature_columns)
    importances[name] = imp
    print(f"=== {name}: TOP 20 feature importances ===")
    print(imp.head(20).to_string(index=False))
    print()

## Redundant-feature check (Section 6.10)

Confirms the importance-splitting effect directly, rather than just asserting it.

In [ ]:
for name, imp in importances.items():
    s = imp.set_index("feature")["importance"]
    print(f"{name}:")
    print(f"  type_TRANSFER={s.get('type_TRANSFER', 0):.4f}  is_transfer={s.get('is_transfer', 0):.4f}  (same underlying signal, split across two columns)")
    print(f"  type_CASH_OUT={s.get('type_CASH_OUT', 0):.4f}  is_cash_out={s.get('is_cash_out', 0):.4f}  (same underlying signal, split across two columns)")
    print(f"  is_transfer_or_cash_out={s.get('is_transfer_or_cash_out', 0):.4f}  (a third, overlapping encoding of the same information)\n")
print("No features are removed in this step -- feature-removal experiments would need a separate controlled comparison.")

## 13. Logistic Regression vs Random Forest comparison

In [ ]:
with open("../results/logistic_regression_metrics.json") as f:
    lr_metrics = json.load(f)

model_specs = [
    ("Logistic Regression - unweighted", "Logistic Regression", None, lr_metrics["model_a_unweighted"]),
    ("Logistic Regression - balanced", "Logistic Regression", "balanced", lr_metrics["model_b_balanced"]),
    ("Random Forest - unweighted", "Random Forest", None, results["model_c_unweighted"]),
    ("Random Forest - balanced_subsample", "Random Forest", "balanced_subsample", results["model_d_balanced_subsample"]),
]
rows = []
for model_name, family, cw, m in model_specs:
    vm, tm = m["validation_metrics"], m["test_metrics"]
    rows.append({
        "model": model_name, "model_family": family, "class_weight": cw,
        "validation_precision": vm["precision"], "validation_recall": vm["recall"], "validation_f1": vm["f1"],
        "validation_roc_auc": vm["roc_auc"], "validation_pr_auc": vm["pr_auc"],
        "test_precision": tm["precision"], "test_recall": tm["recall"], "test_f1": tm["f1"],
        "test_roc_auc": tm["roc_auc"], "test_pr_auc": tm["pr_auc"],
    })
comparison = pd.DataFrame(rows)
comparison

**Observed, factually:** both Random Forest models score dramatically higher than both Logistic Regression models on every metric, including near-1.0 ROC-AUC/PR-AUC and near-1.0 precision AND recall simultaneously at threshold 0.5 -- something Logistic Regression could not achieve at any single threshold in Step 5. This is described factually here; section 15 explains why it should not be read as "Random Forest is simply the better model for this problem."

In [ ]:
lr_at_k = pd.read_csv("../results/logistic_regression_precision_recall_at_k.csv")
rf_at_k_rows = []
for name in fitted:
    for split_name, records in [("validation", results[name]["at_k_validation"]), ("test", results[name]["at_k_test"])]:
        for rec in records:
            rf_at_k_rows.append({"model": name, "split": split_name, **rec})
rf_at_k = pd.DataFrame(rf_at_k_rows)

name_map = {
    "model_a_unweighted": "Logistic Regression - unweighted",
    "model_b_balanced": "Logistic Regression - balanced",
    "model_c_unweighted": "Random Forest - unweighted",
    "model_d_balanced_subsample": "Random Forest - balanced_subsample",
}
combined_at_k = pd.concat([lr_at_k, rf_at_k], ignore_index=True)
combined_at_k["model"] = combined_at_k["model"].map(name_map)
combined_at_k[combined_at_k["k"].isin([100, 1000, 10000])]

## 14. Computational considerations

Both forests (200 trees, max_depth=20, ~4.46M training rows, 8 CPU cores) trained in 178-208 seconds each -- no memory or runtime problems occurred, so the specified configuration was used as-is with no reduction in `n_estimators` and no subsampling of the training data. Mean realized tree depth was ~19.8-19.97 (very close to the `max_depth=20` cap), meaning most trees are using close to their full depth budget -- a hint that depth is a binding constraint worth revisiting if this model family is tuned in a later, separate step.

## 15. Limitations

**This is the most important section in this notebook.** The near-perfect Random Forest metrics (ROC-AUC ~1.0000, PR-AUC ~0.9997-1.0000, precision AND recall both ~0.99+ at threshold 0.5) should be treated with suspicion, not celebrated as a strong generalizable fraud detector. Feature importance confirms why: `amount_to_sender_balance` and `amount_exceeds_sender_balance` dominate both models' importance (45%+ combined for Model C). These are exactly the engineered features built around the Step 2/3 "drained sender account" pattern -- a near-deterministic artifact of how PaySim's fraud-injection logic works (simulated fraud almost always withdraws EXACTLY the available balance), not necessarily a pattern a real fraudster would reliably leave. A tree model can threshold on this ratio far more precisely than Logistic Regression's single linear coefficient could, which is *why* Random Forest looks so much better here -- it isn't that Random Forest found deeper, more meaningful fraud behavior; it's better at exploiting a synthetic-data quirk we already flagged as a leakage/artifact concern back in Step 2.

Other limitations:
- No hyperparameter tuning was performed (by design, per this step's scope) -- `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, and class weights are all fixed starting values, not validated choices.
- Impurity-based feature importance is biased toward continuous features and splits credit across correlated encodings (`type_TRANSFER` vs `is_transfer`), as shown directly above.
- The test-period distribution shift (fraud rate 0.4361% vs train's 0.0816%) documented in Steps 2/4 still applies; test metrics here are not "real-world" performance.
- Reloaded model predictions matched the original in-session predictions to within 2.22e-16 (machine epsilon) rather than being bit-for-bit identical -- a benign, well-known floating-point non-associativity effect of parallel (`n_jobs=-1`) probability averaging across trees, not a real inconsistency. Classifications at threshold 0.5 were 100% identical before vs. after reload.

## 16. Final baseline summary

In [ ]:
import os

joblib.dump(fitted["model_c_unweighted"]["pipeline"], "../models/random_forest_baseline.joblib")
joblib.dump(fitted["model_d_balanced_subsample"]["pipeline"], "../models/random_forest_balanced.joblib")
print("Saved models/random_forest_baseline.joblib and models/random_forest_balanced.joblib")

for name, path in [
    ("model_c_unweighted", "../models/random_forest_baseline.joblib"),
    ("model_d_balanced_subsample", "../models/random_forest_balanced.joblib"),
]:
    reloaded = joblib.load(path)
    reloaded_proba = reloaded.predict_proba(X_val)[:, 1]
    max_diff = np.max(np.abs(fitted[name]["proba_val"] - reloaded_proba))
    same_class = np.array_equal((fitted[name]["proba_val"] >= 0.5).astype(int), (reloaded_proba >= 0.5).astype(int))
    print(f"{name}: max probability diff after reload = {max_diff:.2e} (machine epsilon), classifications identical = {same_class}")

print("\nLogistic Regression artifacts untouched:")
for f in ["../models/logistic_regression_baseline.joblib", "../models/logistic_regression_balanced.joblib"]:
    print(f"  {f}: exists = {os.path.exists(f)}")

In [ ]:
metrics_out = {
    name: {
        "class_weight": ("balanced_subsample" if "balanced" in name else None),
        "training_time_sec": fitted[name]["training_time_sec"],
        "tree_depth": depths[name],
        "validation_metrics": results[name]["validation_metrics"],
        "test_metrics": {**results[name]["test_metrics"], "label": "FINAL TEST RESULTS - not used for model selection"},
        "at_k_validation": results[name]["at_k_validation"],
        "at_k_test": results[name]["at_k_test"],
    }
    for name in fitted
}
metrics_out["_meta"] = {
    "threshold_used": 0.5,
    "configuration": config,
    "no_hyperparameter_tuning": True,
    "important_caveat": (
        "Both Random Forest models achieve near-perfect metrics. This reflects the model's ability to exploit "
        "the 'drained sender account' simulation artifact (Steps 2/3), not a generalizable fraud-detection "
        "breakthrough. See notebook section 15."
    ),
    "test_set_warning": (
        "Test fraud rate (0.4361%) is far higher than train (0.0816%) due to the documented late-period "
        "legitimate-volume collapse. Test metrics were not used to select between Model C and Model D."
    ),
}
with open("../results/random_forest_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

imp_frames = []
for name, imp in importances.items():
    c = imp.copy()
    c.insert(0, "model", name)
    imp_frames.append(c)
pd.concat(imp_frames, ignore_index=True).to_csv("../results/random_forest_feature_importance.csv", index=False)

rf_at_k.to_csv("../results/random_forest_precision_recall_at_k.csv", index=False)
comparison.to_csv("../results/model_baseline_comparison.csv", index=False)
combined_at_k_full = pd.concat([lr_at_k.assign(model=lr_at_k["model"].map(name_map)), rf_at_k.assign(model=rf_at_k["model"].map(name_map))], ignore_index=True)
combined_at_k_full.to_csv("../results/model_baseline_precision_recall_at_k.csv", index=False)

print("Saved results/random_forest_metrics.json")
print("Saved results/random_forest_feature_importance.csv")
print("Saved results/random_forest_precision_recall_at_k.csv")
print("Saved results/model_baseline_comparison.csv")
print("Saved results/model_baseline_precision_recall_at_k.csv")

**Summary:** two Random Forest baselines (unweighted, `balanced_subsample`) were trained on the same 4,463,587-row train period and 42 causal features as Step 5, with no scaling and no hyperparameter search. Both dramatically outperform the Logistic Regression baselines on every metric -- but the dominant driver is the `amount_to_sender_balance`/`amount_exceeds_sender_balance` pair, which encodes a known PaySim simulation artifact rather than confirmed generalizable fraud behavior. No model has been selected as final; no threshold has been optimized; test results were not used to choose between Model C and Model D.